# Clean up

**What it does**
Deletes what demos and test runs leave behind, so the next demo starts from a clean slate: the SQLite database, files converted from evidence, the fake sample files, demo leftovers and, if you choose, stored cases and logs. It first **lists** everything it would delete with sizes, and deletes only after you set `CONFIRM = True`.

**Why**
The demo notebooks (01 to 09) create cases, database rows, converted files and fake evidence. Running them again on top of old results makes the demo confusing. The logic is in the `CleanupService` (`services/maintenance/cleanup_service.py`).

**How to use**
1. Run the Setup cell.
2. In **Step 1** switch the groups on or off (`True` / `False`).
3. Run **Step 2**: it shows the list and deletes nothing.
4. Check the list. If it is right, set `CONFIRM = True` in **Step 3** and run it.
5. Restart the kernels of other open notebooks (Kernel > Restart Kernel) and the API, if running, so nothing keeps using the deleted database.

**Never touched**: `.env`, the code, prompts, a BigQuery dataset, a GCS bucket, and anything outside the storage and evidences folders.

## Setup

Finds the project folder, reads `.env`, and shows which folders and database the cleanup works on.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.config import load_env_file, settings
from services.maintenance.cleanup_service import GROUPS, CleanupService

load_env_file()
config = settings()
cleanup = CleanupService(config)

print("Storage folder:  ", config.storage_root)
print("Evidences folder:", config.evidences_root)
print("Database:        ", config.db_driver, config.db_path if config.db_driver == "sqlite" else "")

## Step 1: Choose what to clean

`True` = include. The safe defaults clean everything the demos create and keep what may be real:
- `cases` is off: `uploads/` can hold real control details and test plan PDFs you uploaded (for example `case-001`).
- `logs` is off: useful when something went wrong.
- `evidence_all` is off: it deletes real evidence too. Use it only when the folder holds nothing but demo files.

In [ ]:
CLEAN = {
    "database": True,          # SQLite file: all case interpretations and test steps
    "evidence_outputs": True,  # PDFs from Word, .xlsx.md / .msg.md texts, pictures, email attachments
    "evidence_samples": True,  # fake files from the factories (*_SAMPLE.*)
    "demo_files": True,        # storage/demo/, sqlite_test.sqlite, api_from_notebook.log
    "reports": True,           # assessment PDFs: outputs/<case>/assessment_*.pdf
    "imports": False,          # uploaded ZIPs + sorted copies (storage/imports/); real uploads may be there
    "cases": False,            # uploads/ and outputs/: ALL stored cases, real uploads included
    "logs": False,             # storage/logs/
    "evidence_all": False,     # EVERYTHING in the evidences folder, real evidence included
}

for group, on in CLEAN.items():
    print(f"{'✔' if on else ' '} {group:<17} {GROUPS[group]}")

## Step 2: What would be deleted (nothing is deleted here)

Every file and folder in the chosen groups, with its size. "SKIPPED" lines are never deleted (cloud storage, memory database).

In [ ]:
plan = cleanup.plan([group for group, on in CLEAN.items() if on])

if not plan:
    print("✅ Nothing to clean")
for target in plan:
    print(target)
total = sum(t.size for t in plan if not t.skipped)
print(f"\n{sum(1 for t in plan if not t.skipped)} item(s), {total / 1024 / 1024:.2f} MB")

## Step 3: Delete

Set `CONFIRM = True` and run this cell to delete exactly the list from Step 2. Run Step 2 again first if you changed Step 1.

In [ ]:
CONFIRM = False

if not CONFIRM:
    print("Nothing deleted. Check the list above, then set CONFIRM = True and run this cell again.")
else:
    deleted = cleanup.run(plan)
    for target in deleted:
        print("🗑", target.path)
    print(f"\n✅ Deleted {len(deleted)} item(s). Restart the kernels of other open notebooks and the API.")

## Step 4: Check

Runs the plan again: after a clean-up it should list nothing (or only SKIPPED lines).

In [ ]:
left = [t for t in cleanup.plan([group for group, on in CLEAN.items() if on]) if not t.skipped]
if not CONFIRM:
    print(f"Not cleaned yet (CONFIRM is False): {len(left)} item(s) waiting.")
else:
    print("✅ Clean" if not left else "Still there:\n" + "\n".join(str(t) for t in left))
